# A/B Testing - L'Oreal India Example

This notebook demonstrates A/B test analysis for testing product reviews feature on L'Oreal India e-commerce.

**Scenario**: Should we add customer reviews to product pages?

**Key Learning**: A/B testing is the gold standard for establishing causation through randomization.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Experiment Setup

**Hypothesis**: Adding product reviews will increase conversion rate.

- **Control (A)**: Current product page without reviews
- **Treatment (B)**: Product page with customer reviews section
- **Primary Metric**: Conversion rate (purchases / visitors)
- **Randomization**: 50/50 split of website visitors

In [ ]:
def generate_ab_test_data(n_control=5000, n_treatment=5000, seed=42):
    """
    Generate synthetic A/B test data.
    
    True effect: Reviews increase conversion by 20% relative (2.0% -> 2.4%)
    """
    np.random.seed(seed)
    
    # Control group (no reviews)
    control_conversion_rate = 0.020  # 2% baseline
    control_conversions = np.random.binomial(1, control_conversion_rate, n_control)
    
    # Treatment group (with reviews) - true lift of 20%
    treatment_conversion_rate = 0.024  # 2.4% with reviews
    treatment_conversions = np.random.binomial(1, treatment_conversion_rate, n_treatment)
    
    control_df = pd.DataFrame({
        'group': 'control',
        'converted': control_conversions,
        'visitor_id': range(n_control)
    })
    
    treatment_df = pd.DataFrame({
        'group': 'treatment',
        'converted': treatment_conversions,
        'visitor_id': range(n_control, n_control + n_treatment)
    })
    
    return pd.concat([control_df, treatment_df], ignore_index=True)

# Generate data
df = generate_ab_test_data()

print("A/B Test Data Summary")
print("=" * 50)
print(f"Control group: {len(df[df['group']=='control']):,} visitors")
print(f"Treatment group: {len(df[df['group']=='treatment']):,} visitors")
print(f"\nSample data:")
df.sample(5)

## 2. Calculate Conversion Rates

In [ ]:
# Aggregate by group
summary = df.groupby('group').agg(
    visitors=('visitor_id', 'count'),
    conversions=('converted', 'sum'),
    conversion_rate=('converted', 'mean')
).round(4)

print("Conversion Summary by Group")
print("=" * 50)
print(summary)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Conversion counts
colors = ['#3498db', '#2ecc71']
axes[0].bar(['Control', 'Treatment'], summary['conversions'], color=colors)
axes[0].set_ylabel('Number of Conversions')
axes[0].set_title('Total Conversions by Group')
for i, v in enumerate(summary['conversions']):
    axes[0].text(i, v + 5, str(int(v)), ha='center', fontsize=12)

# Conversion rates
axes[1].bar(['Control', 'Treatment'], summary['conversion_rate'] * 100, color=colors)
axes[1].set_ylabel('Conversion Rate (%)')
axes[1].set_title('Conversion Rate by Group')
for i, v in enumerate(summary['conversion_rate'] * 100):
    axes[1].text(i, v + 0.05, f'{v:.2f}%', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

## 3. Statistical Analysis

We'll perform a **two-proportion z-test** to determine if the difference is statistically significant.

In [ ]:
def run_proportion_test(df):
    """Run two-proportion z-test."""
    control = df[df['group'] == 'control']['converted']
    treatment = df[df['group'] == 'treatment']['converted']
    
    # Counts
    x1, n1 = control.sum(), len(control)
    x2, n2 = treatment.sum(), len(treatment)
    
    # Proportions
    p1 = x1 / n1
    p2 = x2 / n2
    
    # Pooled proportion
    p_pooled = (x1 + x2) / (n1 + n2)
    
    # Standard error
    se = np.sqrt(p_pooled * (1 - p_pooled) * (1/n1 + 1/n2))
    
    # Z-score
    z = (p2 - p1) / se
    
    # P-value (two-tailed)
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    
    return {
        'control_rate': p1,
        'treatment_rate': p2,
        'absolute_lift': p2 - p1,
        'relative_lift': (p2 - p1) / p1,
        'z_score': z,
        'p_value': p_value
    }

results = run_proportion_test(df)

print("Statistical Test Results")
print("=" * 50)
print(f"\nControl conversion rate:   {results['control_rate']:.2%}")
print(f"Treatment conversion rate: {results['treatment_rate']:.2%}")
print(f"\nAbsolute lift: {results['absolute_lift']:.2%} ({results['absolute_lift']*100:.2f} pp)")
print(f"Relative lift: {results['relative_lift']:.1%}")
print(f"\nZ-score: {results['z_score']:.3f}")
print(f"P-value: {results['p_value']:.4f}")

## 4. Confidence Interval

In [ ]:
def calculate_confidence_interval(df, confidence=0.95):
    """Calculate confidence interval for the difference."""
    control = df[df['group'] == 'control']['converted']
    treatment = df[df['group'] == 'treatment']['converted']
    
    p1, n1 = control.mean(), len(control)
    p2, n2 = treatment.mean(), len(treatment)
    
    diff = p2 - p1
    
    # Standard error of difference
    se_diff = np.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    
    # Z-value for confidence level
    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    
    ci_lower = diff - z * se_diff
    ci_upper = diff + z * se_diff
    
    return ci_lower, ci_upper, se_diff

ci_lower, ci_upper, se = calculate_confidence_interval(df)

print("Confidence Interval")
print("=" * 50)
print(f"\n95% CI for lift: [{ci_lower:.2%}, {ci_upper:.2%}]")
print(f"Standard Error: {se:.4f}")

# Visualize CI
fig, ax = plt.subplots(figsize=(10, 4))

lift = results['absolute_lift']
ax.errorbar(lift * 100, 0, xerr=[[lift*100 - ci_lower*100], [ci_upper*100 - lift*100]], 
            fmt='o', markersize=10, capsize=10, capthick=2, color='green')
ax.axvline(x=0, color='red', linestyle='--', label='No Effect')
ax.set_xlabel('Lift (Percentage Points)')
ax.set_title('95% Confidence Interval for Treatment Effect')
ax.set_yticks([])
ax.legend()
ax.set_xlim(-0.5, 1.0)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print(f"The CI [{ci_lower:.2%}, {ci_upper:.2%}] does NOT include 0.")
print("Therefore, the effect is statistically significant at 95% confidence.")

## 5. Decision Making

In [ ]:
alpha = 0.05  # Significance level

print("Decision Framework")
print("=" * 50)
print(f"\nSignificance level (alpha): {alpha}")
print(f"P-value: {results['p_value']:.4f}")

if results['p_value'] < alpha:
    print(f"\nResult: STATISTICALLY SIGNIFICANT (p < {alpha})")
    print("\nRecommendation: SHIP the reviews feature!")
else:
    print(f"\nResult: NOT statistically significant (p >= {alpha})")
    print("\nRecommendation: Do NOT ship - need more data or no real effect")

# Business impact
print("\n" + "=" * 50)
print("Business Impact Estimate")
print("=" * 50)

monthly_visitors = 500000
current_conversions = monthly_visitors * results['control_rate']
new_conversions = monthly_visitors * results['treatment_rate']
incremental = new_conversions - current_conversions

print(f"\nMonthly visitors: {monthly_visitors:,}")
print(f"Current conversions (without reviews): {current_conversions:,.0f}")
print(f"Expected conversions (with reviews): {new_conversions:,.0f}")
print(f"Incremental conversions: {incremental:,.0f}/month")

avg_order_value = 1500  # Rs
incremental_revenue = incremental * avg_order_value
print(f"\nIncremental monthly revenue: Rs {incremental_revenue:,.0f}")
print(f"Incremental annual revenue: Rs {incremental_revenue * 12:,.0f}")

## 6. Power Analysis (Sample Size Calculation)

Before running an A/B test, we should calculate the required sample size.

In [ ]:
def calculate_sample_size(baseline_rate, mde_relative, alpha=0.05, power=0.80):
    """
    Calculate required sample size per group.
    
    baseline_rate: Current conversion rate
    mde_relative: Minimum detectable effect (relative, e.g., 0.20 for 20% lift)
    """
    p1 = baseline_rate
    p2 = baseline_rate * (1 + mde_relative)
    
    # Effect size (Cohen's h)
    effect_size = 2 * (np.arcsin(np.sqrt(p2)) - np.arcsin(np.sqrt(p1)))
    
    # Z-values
    z_alpha = stats.norm.ppf(1 - alpha/2)
    z_beta = stats.norm.ppf(power)
    
    # Sample size per group
    n = 2 * ((z_alpha + z_beta) / effect_size) ** 2
    
    return int(np.ceil(n))

# Calculate for different MDEs
print("Sample Size Requirements")
print("=" * 50)
print(f"Baseline conversion rate: 2%")
print(f"Significance level: 5%")
print(f"Power: 80%")
print("\n")

mdes = [0.10, 0.15, 0.20, 0.25, 0.30]
for mde in mdes:
    n = calculate_sample_size(0.02, mde)
    print(f"To detect {mde:.0%} relative lift: {n:,} per group ({2*n:,} total)")

## Key Takeaways

1. **A/B testing establishes causation** through random assignment
2. **Statistical significance** (p < 0.05) indicates the effect is unlikely due to chance
3. **Confidence intervals** provide range of plausible effect sizes
4. **Practical significance** matters too - is the effect worth implementing?
5. **Calculate sample size in advance** to ensure adequate power
6. **Don't peek at results** before planned duration - increases false positives